# Run Assignment 2
Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

It asks for Google Drive access and your W&B API key (https://wandb.ai/authorize). Training the baseline takes the longest; the sweeps follow. Results are copied to `MyDrive/CS25M540_SEDL_Assignment2_MobileNetV2`. If the session disconnects, finished W&B runs are kept and the baseline checkpoint is already on Drive.

In [ ]:
# MTech IIT Madras || System Engineering for Deep Learning || Assignment 2
# Created by Tanuj Gupta (CS25M540)
import os
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU, then Run all again.'
print(torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/CS25M540_SEDL_Assignment2_MobileNetV2'
os.makedirs(OUT, exist_ok=True)

In [ ]:
%cd /content
!test -d CS25M540_SEDL_Assignment2_MobileNetV2 || git clone https://github.com/tanujgupta12/CS25M540_SEDL_Assignment2_MobileNetV2.git
%cd /content/CS25M540_SEDL_Assignment2_MobileNetV2
!pip install -q -r requirements.txt

In [ ]:
# Paste your W&B API key when asked
!wandb login

In [ ]:
# Q1: fp32 baseline (reuses the copy on Drive if a previous session already trained it)
if os.path.exists(f'{OUT}/baseline/best.pt'):
    !mkdir -p checkpoints && cp -r "$OUT"/baseline checkpoints/
else:
    !python train.py --epochs 200 --save_dir checkpoints/baseline
    !cp -r checkpoints/baseline "$OUT"/
!python evaluate.py --checkpoint checkpoints/baseline/best.pt | tee checkpoints/baseline/evaluate.txt

In [ ]:
# Q3: PTQ grid (30 runs) and QAT grid (12 runs)
!bash run_sweeps.sh CS6886W-SEDL-Assignment2-MobileNetV2

In [ ]:
# Q4: re-create the best configuration (max compression with <= 1 point accuracy drop) and keep its packed model
best = !python wandb_results.py best --max_drop 1.0
print('
'.join(best))
!python test.py {best[-1]} --save_dir checkpoints/best_compressed
!python evaluate.py --compressed checkpoints/best_compressed/compressed_model.pt | tee checkpoints/best_compressed/evaluate.txt
!cp -r checkpoints/best_compressed "$OUT"/

In [ ]:
# Tables / figures for the report, plus the environment
!python wandb_results.py export --out results.csv
!python wandb_results.py plot --out parallel_coords.png
!pip freeze | grep -iE '^(torch|torchvision|wandb|numpy|matplotlib)==' > environment.txt
!python --version >> environment.txt
!nvidia-smi -L >> environment.txt
!cp results.csv parallel_coords.png environment.txt "$OUT"/
!cat environment.txt